In [2]:
import sys
print(sys.executable)

c:\Users\DELL\Documents\CredResolve Collections Recovery Analytics\.venv\Scripts\python.exe


In [3]:
import statsmodels

print(statsmodels.__version__)

0.14.6


In [4]:
# ============================================================
# CREDRESOLVE — COLLECTIONS RECOVERY ANALYTICS
# 06 — STATISTICAL ANALYSIS
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import warnings

from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")

# ============================================================
# 1. PATHS
# ============================================================

PROJECT_ROOT = Path.cwd().parent
GOLDEN_DIR = PROJECT_ROOT / "data" / "golden"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "tables"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 90)
print("CREDRESOLVE — STATISTICAL ANALYSIS")
print("=" * 90)


# ============================================================
# 2. LOAD GOLDEN DATA
# ============================================================

golden_files = sorted(GOLDEN_DIR.glob("*_golden.csv"))

golden = {
    file.stem.replace("_golden", ""):
    pd.read_csv(file)
    for file in golden_files
}

print("Golden datasets loaded:", len(golden))

required = [
    "accounts",
    "payments",
    "calls",
    "call_attempts",
    "call_dispositions",
    "daily_targeting",
    "campaigns"
]

missing = [
    x for x in required
    if x not in golden
]

if missing:
    raise ValueError(
        f"Missing Golden tables: {missing}"
    )


# ============================================================
# 3. PREPARE ACCOUNT-LEVEL DATASET
# ============================================================

accounts = golden["accounts"].copy()
payments = golden["payments"].copy()
calls = golden["calls"].copy()
attempts = golden["call_attempts"].copy()
dispositions = golden["call_dispositions"].copy()
targeting = golden["daily_targeting"].copy()


# ------------------------------------------------------------
# Payment recovery outcome
# ------------------------------------------------------------

payments["amount"] = pd.to_numeric(
    payments["amount"],
    errors="coerce"
)

payment_status = (
    payments["payment_status"]
    .astype("string")
    .str.lower()
    .str.strip()
)

successful_statuses = {
    "success",
    "successful",
    "completed",
    "paid",
    "settled"
}

payments["successful_payment"] = (
    payment_status.isin(successful_statuses)
)

payment_metrics = (
    payments
    .groupby("account_id")
    .agg(
        successful_payments=(
            "successful_payment",
            "sum"
        ),
        total_payment_amount=(
            "amount",
            "sum"
        ),
        payment_events=(
            "payment_id",
            "nunique"
        )
    )
    .reset_index()
)

payment_metrics["recovered_flag"] = (
    payment_metrics["successful_payments"] > 0
)


# ------------------------------------------------------------
# Call exposure
# ------------------------------------------------------------

calls["duration_sec"] = pd.to_numeric(
    calls["duration_sec"],
    errors="coerce"
)

call_metrics = (
    calls
    .groupby("account_id")
    .agg(
        total_calls=("call_id", "nunique"),
        total_call_duration_sec=(
            "duration_sec",
            "sum"
        ),
        unique_agents=("agent_id", "nunique"),
        unique_campaigns=(
            "campaign_id",
            "nunique"
        ),
        unique_vendors=("vendor_id", "nunique")
    )
    .reset_index()
)


# ------------------------------------------------------------
# Attempts
# ------------------------------------------------------------

attempt_metrics = (
    attempts
    .groupby("account_id")
    .agg(
        total_attempts=(
            "attempt_id",
            "nunique"
        ),
        max_attempt_number=(
            "attempt_no",
            "max"
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# Dispositions
# ------------------------------------------------------------

disposition_metrics = (
    dispositions
    .groupby("account_id")
    .agg(
        disposition_events=(
            "disposition_id",
            "nunique"
        ),
        disposition_codes=(
            "disposition_code",
            "nunique"
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# Targeting
# ------------------------------------------------------------

targeting_metrics = (
    targeting
    .groupby("account_id")
    .agg(
        targeting_events=(
            "target_id",
            "nunique"
        ),
        campaigns_targeted=(
            "campaign_id",
            "nunique"
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# Build analytical dataset
# ------------------------------------------------------------

analysis = accounts.merge(
    payment_metrics,
    on="account_id",
    how="left"
)

analysis = analysis.merge(
    call_metrics,
    on="account_id",
    how="left"
)

analysis = analysis.merge(
    attempt_metrics,
    on="account_id",
    how="left"
)

analysis = analysis.merge(
    disposition_metrics,
    on="account_id",
    how="left"
)

analysis = analysis.merge(
    targeting_metrics,
    on="account_id",
    how="left"
)


# ------------------------------------------------------------
# Fill exposure fields
# ------------------------------------------------------------

numeric_columns = [
    "successful_payments",
    "total_payment_amount",
    "payment_events",
    "total_calls",
    "total_call_duration_sec",
    "unique_agents",
    "unique_campaigns",
    "unique_vendors",
    "total_attempts",
    "max_attempt_number",
    "disposition_events",
    "disposition_codes",
    "targeting_events",
    "campaigns_targeted"
]

for column in numeric_columns:

    if column in analysis.columns:

        analysis[column] = pd.to_numeric(
            analysis[column],
            errors="coerce"
        ).fillna(0)


analysis["recovered_flag"] = (
    analysis["recovered_flag"]
    .fillna(False)
    .astype(int)
)


# ============================================================
# 4. DPD BUCKET
# ============================================================

def dpd_bucket(x):

    if pd.isna(x):
        return "Missing"

    if x <= 0:
        return "Current"

    if x <= 30:
        return "1-30"

    if x <= 60:
        return "31-60"

    if x <= 90:
        return "61-90"

    return "90+"


analysis["dpd_bucket"] = (
    analysis["dpd"]
    .apply(dpd_bucket)
)


# ============================================================
# 5. OVERALL RECOVERY
# ============================================================

n_accounts = analysis["account_id"].nunique()

n_recovered = int(
    analysis["recovered_flag"].sum()
)

overall_rate = (
    n_recovered / n_accounts
    if n_accounts > 0
    else np.nan
)

total_recovery = (
    analysis["total_payment_amount"].sum()
)

overall = pd.DataFrame([{
    "accounts": n_accounts,
    "recovered_accounts": n_recovered,
    "recovery_rate": overall_rate,
    "total_recovery_amount": total_recovery
}])

print("\n" + "=" * 90)
print("OVERALL RECOVERY STATISTICS")
print("=" * 90)

display(overall)

overall.to_csv(
    OUTPUT_DIR /
    "statistical_overall_recovery.csv",
    index=False
)


# ============================================================
# 6. CHI-SQUARE TESTS
# ============================================================

print("\n" + "=" * 90)
print("CATEGORICAL DRIVER TESTS")
print("=" * 90)

categorical_drivers = [
    "loan_type",
    "risk_segment",
    "status",
    "dpd_bucket"
]

chi_results = []

for driver in categorical_drivers:

    table = pd.crosstab(
        analysis[driver],
        analysis["recovered_flag"]
    )

    if (
        table.shape[0] >= 2
        and table.shape[1] >= 2
    ):

        chi2, p_value, dof, expected = (
            stats.chi2_contingency(table)
        )

    else:

        chi2 = np.nan
        p_value = np.nan
        dof = np.nan

    chi_results.append({
        "driver": driver,
        "chi_square": chi2,
        "degrees_of_freedom": dof,
        "p_value": p_value
    })

chi_df = pd.DataFrame(chi_results)

chi_df["significant_5pct"] = (
    chi_df["p_value"] < 0.05
)

display(chi_df)

chi_df.to_csv(
    OUTPUT_DIR /
    "statistical_categorical_tests.csv",
    index=False
)


# ============================================================
# 7. NUMERIC DRIVER TESTS
# ============================================================

print("\n" + "=" * 90)
print("NUMERIC DRIVER TESTS")
print("=" * 90)

numeric_drivers = [
    "dpd",
    "principal_amount",
    "outstanding_amount",
    "total_calls",
    "total_call_duration_sec",
    "total_attempts",
    "targeting_events",
    "campaigns_targeted",
    "disposition_events",
    "unique_agents",
    "unique_vendors"
]

numeric_results = []

for driver in numeric_drivers:

    if driver not in analysis.columns:
        continue

    recovered = (
        analysis.loc[
            analysis["recovered_flag"] == 1,
            driver
        ]
        .dropna()
    )

    not_recovered = (
        analysis.loc[
            analysis["recovered_flag"] == 0,
            driver
        ]
        .dropna()
    )

    if (
        len(recovered) > 1
        and len(not_recovered) > 1
    ):

        statistic, p_value = (
            stats.mannwhitneyu(
                recovered,
                not_recovered,
                alternative="two-sided"
            )
        )

        recovered_median = recovered.median()
        not_recovered_median = not_recovered.median()

    else:

        statistic = np.nan
        p_value = np.nan
        recovered_median = np.nan
        not_recovered_median = np.nan

    numeric_results.append({
        "driver": driver,
        "recovered_median": recovered_median,
        "not_recovered_median": not_recovered_median,
        "median_difference": (
            recovered_median
            -
            not_recovered_median
        ),
        "mann_whitney_u": statistic,
        "p_value": p_value
    })

numeric_df = pd.DataFrame(
    numeric_results
)

numeric_df["significant_5pct"] = (
    numeric_df["p_value"] < 0.05
)

display(numeric_df)

numeric_df.to_csv(
    OUTPUT_DIR /
    "statistical_numeric_tests.csv",
    index=False
)


# ============================================================
# 8. EFFECT SIZE — CLIFF'S DELTA
# ============================================================

print("\n" + "=" * 90)
print("EFFECT SIZE ANALYSIS")
print("=" * 90)


def cliffs_delta(x, y):

    x = np.asarray(x)
    y = np.asarray(y)

    x = x[~pd.isna(x)]
    y = y[~pd.isna(y)]

    if len(x) == 0 or len(y) == 0:
        return np.nan

    comparisons = (
        x[:, None] > y[None, :]
    ).sum()

    reverse = (
        x[:, None] < y[None, :]
    ).sum()

    return (
        comparisons - reverse
    ) / (
        len(x) * len(y)
    )


effect_results = []

for driver in numeric_drivers:

    if driver not in analysis.columns:
        continue

    recovered = (
        analysis.loc[
            analysis["recovered_flag"] == 1,
            driver
        ]
        .dropna()
        .values
    )

    not_recovered = (
        analysis.loc[
            analysis["recovered_flag"] == 0,
            driver
        ]
        .dropna()
        .values
    )

    effect = cliffs_delta(
        recovered,
        not_recovered
    )

    effect_results.append({
        "driver": driver,
        "cliffs_delta": effect,
        "absolute_cliffs_delta":
            abs(effect)
            if pd.notna(effect)
            else np.nan
    })

effect_df = pd.DataFrame(
    effect_results
)

display(effect_df)

effect_df.to_csv(
    OUTPUT_DIR /
    "statistical_effect_sizes.csv",
    index=False
)


# ============================================================
# 9. MULTIPLE TESTING — BONFERRONI
# ============================================================

all_pvalues = pd.concat([
    chi_df["p_value"],
    numeric_df["p_value"]
]).dropna()

number_of_tests = len(all_pvalues)

bonferroni_alpha = (
    0.05 / number_of_tests
    if number_of_tests > 0
    else np.nan
)

chi_df["bonferroni_significant"] = (
    chi_df["p_value"]
    <
    bonferroni_alpha
)

numeric_df["bonferroni_significant"] = (
    numeric_df["p_value"]
    <
    bonferroni_alpha
)

print(
    "Number of statistical tests:",
    number_of_tests
)

print(
    "Bonferroni alpha:",
    bonferroni_alpha
)


# ============================================================
# 10. LOGISTIC REGRESSION
# ============================================================
#
# Outcome:
# recovered_flag
#
# This estimates conditional associations.
# It does NOT establish causality.
# ============================================================

print("\n" + "=" * 90)
print("MULTIVARIABLE LOGISTIC REGRESSION")
print("=" * 90)

model_columns = [
    "recovered_flag",
    "dpd",
    "principal_amount",
    "outstanding_amount",
    "total_calls",
    "total_attempts",
    "targeting_events",
    "loan_type",
    "risk_segment"
]

model_data = analysis[
    model_columns
].copy()

for column in [
    "dpd",
    "principal_amount",
    "outstanding_amount",
    "total_calls",
    "total_attempts",
    "targeting_events"
]:

    model_data[column] = pd.to_numeric(
        model_data[column],
        errors="coerce"
    )

model_data = model_data.dropna()

# Limit extreme monetary values for numerical stability
for column in [
    "principal_amount",
    "outstanding_amount"
]:

    low = model_data[column].quantile(0.01)
    high = model_data[column].quantile(0.99)

    model_data[column] = (
        model_data[column]
        .clip(low, high)
    )

formula = (
    "recovered_flag ~ "
    "dpd + "
    "principal_amount + "
    "outstanding_amount + "
    "total_calls + "
    "total_attempts + "
    "targeting_events + "
    "C(loan_type) + "
    "C(risk_segment)"
)

try:

    model = smf.logit(
        formula=formula,
        data=model_data
    ).fit(
        disp=False,
        maxiter=200
    )

    coefficients = pd.DataFrame({

        "variable":
            model.params.index,

        "coefficient":
            model.params.values,

        "odds_ratio":
            np.exp(model.params.values),

        "p_value":
            model.pvalues.values,

        "ci_lower_odds_ratio":
            np.exp(model.conf_int()[0].values),

        "ci_upper_odds_ratio":
            np.exp(model.conf_int()[1].values)
    })

    coefficients["significant_5pct"] = (
        coefficients["p_value"] < 0.05
    )

    print(model.summary())

    display(coefficients)

    coefficients.to_csv(
        OUTPUT_DIR /
        "statistical_logistic_regression.csv",
        index=False
    )

    model_fit = pd.DataFrame([{
        "observations": int(model.nobs),
        "pseudo_r_squared": model.prsquared,
        "aic": model.aic,
        "bic": model.bic
    }])

    display(model_fit)

    model_fit.to_csv(
        OUTPUT_DIR /
        "statistical_logistic_model_fit.csv",
        index=False
    )

except Exception as error:

    print(
        "Logistic regression could not be fitted:"
    )

    print(error)


# ============================================================
# 11. CALL EXPOSURE STATISTICAL CHECK
# ============================================================

print("\n" + "=" * 90)
print("CALL EXPOSURE ANALYSIS")
print("=" * 90)

analysis["call_exposure_group"] = pd.cut(
    analysis["total_calls"],
    bins=[
        -np.inf,
        2,
        3,
        4,
        np.inf
    ],
    labels=[
        "0-2",
        "3",
        "4",
        "5+"
    ]
)

exposure_summary = (
    analysis
    .groupby(
        "call_exposure_group",
        observed=False
    )
    .agg(
        accounts=(
            "account_id",
            "nunique"
        ),
        recovered_accounts=(
            "recovered_flag",
            "sum"
        ),
        recovery_amount=(
            "total_payment_amount",
            "sum"
        ),
        avg_dpd=(
            "dpd",
            "mean"
        )
    )
    .reset_index()
)

exposure_summary["recovery_rate"] = (
    exposure_summary["recovered_accounts"]
    /
    exposure_summary["accounts"]
)

display(exposure_summary)

exposure_summary.to_csv(
    OUTPUT_DIR /
    "statistical_call_exposure.csv",
    index=False
)


# ============================================================
# 12. STATISTICAL EVIDENCE SUMMARY
# ============================================================

print("\n" + "=" * 90)
print("STATISTICAL EVIDENCE SUMMARY")
print("=" * 90)

evidence_rows = []

for _, row in chi_df.iterrows():

    evidence_rows.append({
        "analysis": "Chi-square",
        "driver": row["driver"],
        "p_value": row["p_value"],
        "significant_5pct":
            row["significant_5pct"],
        "bonferroni_significant":
            row["bonferroni_significant"]
    })

for _, row in numeric_df.iterrows():

    evidence_rows.append({
        "analysis": "Mann-Whitney U",
        "driver": row["driver"],
        "p_value": row["p_value"],
        "significant_5pct":
            row["significant_5pct"],
        "bonferroni_significant":
            row["bonferroni_significant"]
    })

evidence_df = pd.DataFrame(
    evidence_rows
)

display(evidence_df)

evidence_df.to_csv(
    OUTPUT_DIR /
    "statistical_evidence_summary.csv",
    index=False
)


# ============================================================
# 13. FINAL SUMMARY
# ============================================================

significant_5pct = int(
    evidence_df[
        "significant_5pct"
    ]
    .fillna(False)
    .sum()
)

significant_bonferroni = int(
    evidence_df[
        "bonferroni_significant"
    ]
    .fillna(False)
    .sum()
)

final_summary = pd.DataFrame([{

    "accounts_analyzed":
        n_accounts,

    "recovered_accounts":
        n_recovered,

    "overall_recovery_rate":
        overall_rate,

    "total_recovery_amount":
        total_recovery,

    "statistical_tests_run":
        number_of_tests,

    "significant_at_5pct":
        significant_5pct,

    "significant_after_bonferroni":
        significant_bonferroni,

    "causal_claims_made":
        False
}])

print("\n" + "=" * 90)
print("STATISTICAL ANALYSIS SUMMARY")
print("=" * 90)

display(final_summary)

final_summary.to_csv(
    OUTPUT_DIR /
    "statistical_analysis_summary.csv",
    index=False
)


# ============================================================
# 14. SAVE ANALYTICAL DATASET
# ============================================================

analysis.to_csv(
    OUTPUT_DIR /
    "statistical_account_level_dataset.csv",
    index=False
)


# ============================================================
# 15. FINAL STATUS
# ============================================================

print("\n" + "=" * 90)
print("STATISTICAL ANALYSIS COMPLETE")
print("=" * 90)

print(
    "Golden Dataset used as analytical source."
)

print(
    "Raw source files were NOT modified."
)

print(
    "Statistical significance does NOT imply causality."
)

print(
    "Effect sizes and multiple-testing adjustment included."
)

print(
    f"Outputs saved to: {OUTPUT_DIR}"
)

CREDRESOLVE — STATISTICAL ANALYSIS
Golden datasets loaded: 18

OVERALL RECOVERY STATISTICS


,accounts,recovered_accounts,recovery_rate,total_recovery_amount
0,30000,13284,0.4428,1.917259e+09



CATEGORICAL DRIVER TESTS


,driver,chi_square,degrees_of_freedom,p_value,significant_5pct
0,loan_type,4.101268,4,0.392474,False
1,risk_segment,2.678277,3,0.443932,False
2,status,1.155898,3,0.763600,False
3,dpd_bucket,10.887965,4,0.027852,True



NUMERIC DRIVER TESTS


...


STATISTICAL EVIDENCE SUMMARY


,analysis,driver,p_value,significant_5pct,bonferroni_significant
0,Chi-square,loan_type,0.392474,False,False
1,Chi-square,risk_segment,0.443932,False,False
2,Chi-square,status,0.763600,False,False
3,Chi-square,dpd_bucket,0.027852,True,False
4,Mann-Whitney U,dpd,0.997056,False,False
5,Mann-Whitney U,principal_amount,0.929304,False,False
6,Mann-Whitney U,outstanding_amount,0.888020,False,False
7,Mann-Whitney U,total_calls,0.101738,False,False
8,Mann-Whitney U,total_call_duration_sec,0.333138,False,False
9,Mann-Whitney U,total_attempts,0.334832,False,False



STATISTICAL ANALYSIS SUMMARY


,accounts_analyzed,recovered_accounts,overall_recovery_rate,total_recovery_amount,statistical_tests_run,significant_at_5pct,significant_after_bonferroni,causal_claims_made
0,30000,13284,0.4428,1.917259e+09,15,2,0,False



STATISTICAL ANALYSIS COMPLETE
Golden Dataset used as analytical source.
Raw source files were NOT modified.
Statistical significance does NOT imply causality.
Effect sizes and multiple-testing adjustment included.
Outputs saved to: c:\Users\DELL\Documents\CredResolve Collections Recovery Analytics\outputs\tables
